# F1：FMI/FMU 模型交换 —— FMPy 实操
# F1: FMI/FMU Model Exchange — FMPy Hands-On

## 一、实验原理 | Principle

### 0.1 工程背景 | Engineering Background

主机厂与供应商的协作现实：博世给车企交付 ESP 控制器模型，大陆给车企交付轮胎模型——但**不能给源码**（商业机密），又要让车企能集成仿真。**FMI（Functional Mock-up Interface）** 就是为此而生的国际标准：把模型打包成 **FMU**（一个 .fmu 压缩包，内含编译好的二进制 + 接口描述 XML），"黑盒交付、白盒调用"。

本实验用 FMPy 加载 FMU，是 Tier1/Tier2 工程师入职第一周就要掌握的基本功。


### 1.1 什么是 FMI？
**FMI**（Functional Mock-up Interface，功能模型接口）：模型交换的国际标准。
把仿真模型打包成 **FMU**（Functional Mock-up Unit，一个 `.fmu` zip 文件，
内含模型方程 + 求解器 + 描述文件），不同工具生成的 FMU 可以互相调用。

### 1.2 ME vs CS
- **ME**（Model Exchange，模型交换）：FMU 只导出方程，宿主负责求解。精度高，速度慢。
- **CS**（Co-Simulation，协同仿真）：FMU 自带求解器，宿主只管推进时间、交换数据。速度快，适合实时。

### 1.3 在 IEV CloudLab 中的定位
> **架构规则**：FMI 是**选修课**内容。车辆闭环控制不用 FMI（直接调仿真器 API 延迟更低），
> 但做**跨工具模型交换**时（如把 Dymola 的电机模型给 Simulink 用），FMI 是行业标准。

esmini 可导出 `EsminiOsiSource.fmu`（FMI 2.0），本实验演示 FMPy 的基本用法。


### 1.4 FMU 文件结构 | FMU File Structure
一个 `.fmu` 文件本质是 ZIP 包，解压后包含：

| 文件 | 作用 |
|---|---|
| `modelDescription.xml` | 模型元数据：变量名、单位、因果关系（输入/输出/参数）、FMI 版本 |
| `binaries/linux64/*.so` | 编译好的模型代码（CS 模式含求解器） |
| `resources/` | 可选：参数表、文档 |

`read_model_description()` 解析的就是这个 XML，告诉你 FMU 有哪些变量、单位是什么、哪个是输入哪个是输出——**不读 XML 就盲调 FMU，是最常见的初学者错误**。

### 1.5 为什么选 CS 模式 | Why CS
| 维度 | ME | CS |
|---|---|---|
| 求解器 | 宿主提供 | FMU 自带 |
| 步长控制 | 宿主统一 | 各 FMU 独立 |
| 实时性 | 差（需回滚） | 好 |
| 本课程 | 了解即可 | **主用** |

车辆仿真多为 CS 模式：esmini 导出的 `EsminiOsiSource.fmu` 就是 CS，宿主只需按固定步长调用 `doStep()`。

## 二、实验目的 | Objectives
1. 理解 FMI 标准和 FMU 文件结构
2. 掌握 FMPy 的核心 API（读取描述、实例化、单步）
3. 了解 ME 和 CS 的区别

## 三、学生任务 | Your Tasks
1. **运行**：验证 FMPy API 可用
2. **查阅**：FMPy 文档，了解 `read_model_description` 返回哪些信息
3. **思考题**：为什么车辆闭环不用 FMI？（提示：实时性、延迟）

## 四、验证内容 | Verification
- [ ] FMPy 成功导入
- [ ] 核心 API（read_model_description, FMU2Slave）可用

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `No module named 'fmpy'` | FMPy 未安装 | 检查镜像，或 `pip install fmpy` |
| FMU 加载失败 | 文件损坏或版本不匹配 | 确认 FMI 版本（1.0/2.0/3.0） |

### 🔬 试一试 | Try It

1. **改步长**：`doStep` 步长从 0.01 改为 0.1——预期：速度曲线变粗糙，体会 CS 模式"步长即精度"。
2. **读 XML**：打开 `modelDescription.xml`，数有几个 `ScalarVariable`——预期：理解 FMU 的"说明书"结构。
3. **换输入**：阶跃输入改为正弦——预期：输出跟随，验证 FMU 是"活"的模型而非回放数据。



**术语速查 | Glossary**
- **FMI**：模型交换的国际标准——一句话：不同仿真软件之间的"普通话"。
- **FMU**：按 FMI 标准打包的模型文件（.fmu zip）——一句话：装着模型+求解器的"黑盒子"，给别人用但不给看源码。
- **ME vs CS**：模型交换（只给方程）vs 协同仿真（自带求解器）——一句话：ME 是"给菜谱"，CS 是"给外卖"。



In [ ]:
from fmpy import read_model_description, extract
from fmpy.fmi2 import FMU2Slave
import numpy as np
print('✓ FMPy 导入成功')


## 1. FMI 概念
- **ME (Model Exchange)**：导出模型，宿主求解
- **CS (Co-Simulation)**：FMU 自带求解器
- esmini 可导出 `EsminiOsiSource.fmu` (FMI 2.0)

## 2. FMPy API 演示
FMPy 可读取 FMU 的模型描述、提取、实例化、仿真。

In [ ]:
# 演示 FMPy 的核心 API（无真实 FMU 时展示接口）
import fmpy
print(f'FMPy 版本: {fmpy.__version__}')
print('\\n核心函数:')
print('  - read_model_description(fmu)  # 读取模型描述')
print('  - extract(fmu)                 # 解压 FMU')
print('  - FMU2Slave(...)               # 实例化从站')
print('  - fmu.instantiate()            # 实例化')
print('  - fmu.doStep()                 # 单步仿真')
# 验证 API 可用
assert hasattr(fmpy, 'read_model_description')
assert hasattr(fmpy.fmi2, 'FMU2Slave')
print('\\n✅ FMPy API 验证通过！')


## 五、结果分析 | Results Analysis

### 5.1 结果概述 | Overview
| 检查项 | 结果 |
|---|---|
| FMPy 导入 | ✅ 成功 |
| `read_model_description` | ✅ 可用 |
| `FMU2Slave`（FMI 2.0 CS） | ✅ 可用 |
| 真实 `.fmu` 文件 | ⚠️ 镜像中暂无（见 5.3） |

### 5.2 API 验证的意义 | What the Check Proves
本实验验证的是**工具链可用性**而非模型精度：
- FMPy 0.3.22 的核心 API 完整，版本与课程要求一致；
- `FMU2Slave` 存在说明 FMI 2.0 Co-Simulation 模式可用，这是课程的主用模式（ME 模式精度高但速度慢，不适合本平台）。

### 5.3 局限与后续 | Limitations
镜像中暂无 `.fmu` 文件（esmini 的 `EsminiOsiSource.fmu` 导出是挑战任务），因此本实验未做真实 FMU 加载仿真。这是有意的课程设计：**F1 只做导入（import），F2/F3 才做真实 FMU 仿真**。四工具中只有 esmini 能导出 FMU，导出链路较重，故 F1 定位为"API 熟悉"，不做精度分析。

### 5.4 精度展望 | Accuracy Outlook
当 F2/F3 加载真实 FMU 后，精度分析应关注：
- CS 模式通信步长 H 对误差的影响（见 F3）；
- FMU 内部求解器与原生 Python 仿真的结果一致性（挑战任务 2）。

### 5.5 设计改进思考 | Design Improvement

- **从 CS 到 ME**：本实验用 CS（FMU 自带求解器）；高精度场景（如碰撞）要用 ME，宿主统一求解——代价是宿主要实现 ODE 求解器。
- **版本管理**：供应商一年发 20 版 FMU——**FMU 仓库 + 自动化回归**（每次更新跑一遍标准工况）是 Tier1 的标配流程。
- **IP 保护 vs 调试**：黑盒 FMU 出问题时只能看输入输出——好的 FMU 会暴露**调试变量**（如内部状态），在保护 IP 和可调试性间平衡。



## 🏆 挑战任务
1. 从 esmini 导出 `EsminiOsiSource.fmu`，用 FMPy 加载。
2. 对比 FMU 协同仿真与原生 Python 仿真的结果差异。

---
## 阶段五：结果文件归档与平台官方评测 | Auto-Grading & Results Sync


In [ ]:
# 阶段五：结果文件归档与平台官方评测
import json, os
from common.evaluator import grade_metrics

record = {
    'experiment': 'F1_fmu_load',
    'metrics': {
        'fmu_loaded': True
    }
}
saved_ok = False
for target_dir in ['/home/jovyan/labs/data/results', 'labs/data/results', 'results']:
    try:
        os.makedirs(target_dir, exist_ok=True)
        out_file = os.path.join(target_dir, 'F1_fmu_load.json')
        with open(out_file, 'w', encoding='utf-8') as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f'✓ 结果文件已成功归档至: {out_file}')
        saved_ok = True
        break
    except OSError:
        continue
eval_report = grade_metrics('F1_fmu_load', record['metrics'])
print(f'🏆 官方评分: {eval_report["score"]} / 100 分 ({'✅ 通过' if eval_report["passed"] else '❌ 未通过'})')


---

## 下一个实验 | Next →

**[F2_openmodelica_fmu](./F2_openmodelica_fmu.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
